### Conditional Order (Makespan)
For makespan minimisation, suppose two separation-identical aircraft $i$ and $j$ satisfy $r_i \leq r_j$ and $lt_i \leq lt_j$. If the following inequality holds, then placing $i$ before $j$ yields a makespan no worse than the swapped sequence.

$$
W_1 (t_i - b_i)^{\alpha} + W_2 C(t_i, lc_i)
+ W_1 (t_j - b_j)^{\alpha} + W_2 C(t_j, lc_j)
\le\ W_1 (t_i^\prime - b_i)^{\alpha} + W_2 C(t_i^\prime, lc_i)
+ W_1 (t_j^\prime - b_j)^{\alpha} + W_2 C(t_j^\prime, lc_j)
$$


In [1]:
from core.context import *
from core.checks import *

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()

# Left and right sides of the new two-aircraft inequality.
def C(t, lc) -> ArithRef: return W2 * If(t <= lc, RealVal(0), If(t <= lc + 300, ω1 * (t - lc) + ω2, ω3 * (t - lc) + ω4))
lhs = W1 * ((S1.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S1.takeoff["i"], ctx.lc["i"]) \
    + W1 * ((S1.takeoff["j"] - ctx.b["j"]) ** α) + W2 * C(S1.takeoff["j"], ctx.lc["j"])
rhs = W1 * ((S2.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S2.takeoff["i"], ctx.lc["i"]) \
    + W1 * ((S2.takeoff["j"] - ctx.b["j"]) ** α) + W2 * C(S2.takeoff["j"], ctx.lc["j"])

# Encode and enforce the rule premises; total order on release/tw times and that i and j are delta-equivalent.
premises = [
    ctx.r["i"] <= ctx.r["j"],  ctx.lt["i"] <= ctx.lt["j"],
    ctx.b["i"] <= ctx.b["j"],  ctx.lc["i"] <= ctx.lc["j"],
    *ctx.separation_equivalence("i", "j"),
    lhs <= rhs
]

# Encode the claim; no greater makespan in S2 than in S1.
claim = S1.makespan <= S2.makespan

# Check the solver result.
res = verify_pruning_rule(ctx, premises, claim)
print(f"Conditional Order Makespan (total):           {res}")


Conditional Order Makespan (total):           Verified


### Conditional Order (Delay & CTOT)
Using the same release-time, latest-time, and separation-equivalence premises, if the following inequality holds, then the combined delay-and-CTOT dominance check already in this notebook follows.

$$
W_1 (t_i - b_i)^{\alpha} + W_2 C(t_i, lc_i)
+ W_1 (t_j - b_j)^{\alpha} + W_2 C(t_j, lc_j)
\le\ W_1 (t_i^\prime - b_i)^{\alpha} + W_2 C(t_i^\prime, lc_i)
+ W_1 (t_j^\prime - b_j)^{\alpha} + W_2 C(t_j^\prime, lc_j)
$$

In [2]:
from core.context import *
from core.checks import *

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()

# Left and right sides of the new two-aircraft inequality.
def C(t, lc) -> ArithRef: return W2 * If(t <= lc, RealVal(0), If(t <= lc + 300, ω1 * (t - lc) + ω2, ω3 * (t - lc) + ω4))
lhs = W1 * ((S1.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S1.takeoff["i"], ctx.lc["i"]) \
    + W1 * ((S1.takeoff["j"] - ctx.b["j"]) ** α) + W2 * C(S1.takeoff["j"], ctx.lc["j"])
rhs = W1 * ((S2.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S2.takeoff["i"], ctx.lc["i"]) \
    + W1 * ((S2.takeoff["j"] - ctx.b["j"]) ** α) + W2 * C(S2.takeoff["j"], ctx.lc["j"])

# Encode and enforce the rule premises; total order on release/tw times and that i and j are delta-equivalent.
premises = [
    ctx.r["i"] <= ctx.r["j"],  ctx.lt["i"] <= ctx.lt["j"],
    ctx.b["i"] <= ctx.b["j"],  ctx.lc["i"] <= ctx.lc["j"],
    *ctx.separation_equivalence("i", "j"),
    lhs <= rhs
]

# Encode the claim; no aircraft is more delayed in S1 than S2 (aside i and j).
claim = And(*[
    S1.ctot[x] + S1.delay[x] <= S2.ctot[x] + S2.delay[x]
    for x in ctx.aircraft if x not in ("i", "j")
])

# Check the solver result.
res = verify_pruning_rule(ctx, premises, claim)
print(f"Conditional Order Delay & CTOT (per not i/j ac):  {res}")

# Encode the claim; no aircraft has a higher CTOT penalty in S1 than S2 (aside i and j).
claim = Sum([S1.ctot[x] for x in ctx.aircraft]) <= Sum([S2.ctot[x] for x in ctx.aircraft])


# Check the solver result.
res = verify_pruning_rule(ctx, premises, claim)
print(f"Conditional Order CTOT (per ac):  {res}")
res.counterexample.print_model(ctx, sequences=[S1, S2])

# Encode the claim; total (delay + ctot) in S1 <= total (delay + ctot) in S2.
claim = (Sum([(S1.ctot[x] + S1.delay[x]) for x in ctx.aircraft])
         <= Sum([(S2.ctot[x] + S2.delay[x]) for x in ctx.aircraft]))

# Check the solver result.
res = verify_pruning_rule(ctx, premises, claim)
print(f"Conditional Order Delay & CTOT (total):           {res}")


Conditional Order Delay & CTOT (per not i/j ac):  Verified
Conditional Order CTOT (per ac):  Unverified(counterexample = yes, vacuous = no)


AttributeError: 'Counterexample' object has no attribute 'print_model'

### Conditional Order (Delay)
For delay-based objectives, under $r_i \leq r_j$, $lt_i \leq lt_j$, and separation equivalence, if the following inequality holds, then placing $i$ before $j$ yields a schedule whose delay is no worse than the swapped sequence, both per aircraft and in total.

$$
W_1 (t_i - b_i)^{\alpha} + W_2 C(t_i, lc_i)
+ W_1 (t_j - b_j)^{\alpha} + W_2 C(t_j, lc_j)
\le\ W_1 (t_i^\prime - b_i)^{\alpha} + W_2 C(t_i^\prime, lc_i)
+ W_1 (t_j^\prime - b_j)^{\alpha} + W_2 C(t_j^\prime, lc_j)
$$


In [7]:
from core.context import *
from core.checks import *
from z3 import Sum

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()

# Left and right sides of the new two-aircraft inequality.
def C(t, lc) -> ArithRef: return W2 * If(t <= lc, RealVal(0), If(t <= lc + 300, ω1 * (t - lc) + ω2, ω3 * (t - lc) + ω4))
lhs = W1 * ((S1.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S1.takeoff["i"], ctx.lc["i"]) \
    + W1 * ((S1.takeoff["j"] - ctx.b["j"]) ** α) + W2 * C(S1.takeoff["j"], ctx.lc["j"])
rhs = W1 * ((S2.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S2.takeoff["i"], ctx.lc["i"]) \
    + W1 * ((S2.takeoff["j"] - ctx.b["j"]) ** α) + W2 * C(S2.takeoff["j"], ctx.lc["j"])

# Encode and enforce the rule premises; total order on release/tw times and that i and j are delta-equivalent.
premises = [
    ctx.r["i"] <= ctx.r["j"],  ctx.lt["i"] <= ctx.lt["j"],
    ctx.b["i"] <= ctx.b["j"],  ctx.lc["i"] <= ctx.lc["j"],
    *ctx.separation_equivalence("i", "j"),
    lhs <= rhs
]

# Verify that every aircraft other than i and j incurs no larger delay in S1.
claim = And(*[
    S1.delay[x] <= S2.delay[x]
    for x in ctx.aircraft if x not in ("i", "j")
])

# Ask the solver to certify the claim.
res = verify_pruning_rule(ctx, premises, claim)
print(f"Conditional Order Delay (per-aircraft): {res}")

# Verify that total delay in S1 is no larger than in S2.
claim = Sum([S1.delay[x] for x in ctx.aircraft]) <= Sum([S2.delay[x] for x in ctx.aircraft])

# Ask the solver to certify the claim.
res = verify_pruning_rule(ctx, premises, claim)
print(f"Conditional Order Delay (total):        {res}")


Conditional Order Delay (per-aircraft): Verified
Conditional Order Delay (total):        Verified


### Conditional Order (Time Window Feasibility)
For time-window feasibility, under the same release-time, latest-time, and separation-equivalence premises, if the following inequality holds and the swapped sequence is feasible, then placing $i$ before $j$ does not introduce any new time-window violation.

$$
W_1 (t_i - b_i)^{\alpha} + W_2 C(t_i, lc_i)
+ W_1 (t_j - b_j)^{\alpha} + W_2 C(t_j, lc_j)
\le\ W_1 (t_i^\prime - b_i)^{\alpha} + W_2 C(t_i^\prime, lc_i)
+ W_1 (t_j^\prime - b_j)^{\alpha} + W_2 C(t_j^\prime, lc_j)
$$

In [8]:
from core.context import *
from core.checks import *
from z3 import Implies

# Retrieve two sequences, only differing by swapping the positions of i and j.
S1, S2, ctx = get_sequences()

# Left and right sides of the new two-aircraft inequality.
def C(t, lc) -> ArithRef: return W2 * If(t <= lc, RealVal(0), If(t <= lc + 300, ω1 * (t - lc) + ω2, ω3 * (t - lc) + ω4))
lhs = W1 * ((S1.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S1.takeoff["i"], ctx.lc["i"]) \
    + W1 * ((S1.takeoff["j"] - ctx.b["j"]) ** α) + W2 * C(S1.takeoff["j"], ctx.lc["j"])
rhs = W1 * ((S2.takeoff["i"] - ctx.b["i"]) ** α) + W2 * C(S2.takeoff["i"], ctx.lc["i"]) \
    + W1 * ((S2.takeoff["j"] - ctx.b["j"]) ** α) + W2 * C(S2.takeoff["j"], ctx.lc["j"])

# State the premises: order the release and latest times, require separation equivalence, prefer S1 by the new inequality, and assume S2 is feasible.
premises = [
    ctx.r["i"] <= ctx.r["j"],  ctx.lt["i"] <= ctx.lt["j"],
    ctx.b["i"] <= ctx.b["j"],  ctx.lc["i"] <= ctx.lc["j"],
    *ctx.separation_equivalence("i", "j"),
    lhs <= rhs,
    *S2.time_window_feasible
]

# Verify that S1 does not introduce any time-window violation absent from S2.
claim = And(*[
    Implies(S2.window_violation[x], S1.window_violation[x])
    for x in ctx.aircraft
])

# Ask the solver to certify the claim.
res = verify_pruning_rule(ctx, premises, claim)
print(f"Conditional Order Time Windows: {res}")


Conditional Order Time Windows: Verified
